In [ ]:
import sys
from rad_ai import query
import time
import sys; sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1")
from benchmark_golden_pairs_1 import golden_pairs, should_refuse_map, answer_keys
'''
Quality:
  CoP - Context Precision: fraction of retrieved chunks that are golden
  CiP - Citation Precision: fraction of cited sources that are correct
  RR  - Refusal Rate: fraction of correct refusal decisions (maximize)
  HR  - Hallucination Rate: fraction of answers with unsupported claims (minimize)
Latency:
  RT  - mean retrieval time (s)
'''


def get_citation(hit) -> str:
    return f'{hit["source"]} | page {hit["page"]}'

cid2sp   = {m["chunk_id"]: f'{m["source"]} | page {m["page"]}' for m in metas}
golden_sp = {q: {cid2sp[c] for c in ids} for q, ids in golden_pairs.items()}

def context_precision(query, retrieved_hits):
    if not retrieved_hits: return 0.0
    g = golden_sp.get(query, set())
    return sum(1 for h in retrieved_hits if get_citation(h) in g) / len(retrieved_hits)

def context_recall(query, retrieved_hits):        # Hit@k — the honest single-golden signal
    g = golden_sp.get(query, set())
    if not g: return None
    got = {get_citation(h) for h in retrieved_hits}
    return len(got & g) / len(g)

def citation_precision(cited_ids, golden_ids):    # cited_ids already (source,page) strings
    if not cited_ids: return 0.0
    return sum(1 for c in cited_ids if c in golden_ids) / len(cited_ids)

def refusal_score(did_refuse, should_refuse):
    correct = sum(1 for d, s in zip(did_refuse, should_refuse) if d == s)
    return correct / len(should_refuse)

def hallucination_rate(judgments):
    """judgments: list of bools, True = answer contained hallucination.
    v0: fill by hand or with an LLM judge."""
    if not judgments:
        return 0.0
    return sum(judgments) / len(judgments)

def quality_score(s):
    return (0.25 * s['COP']
          + 0.25 * s['CIP']
          + 0.25 * s['RR']          # already "higher = better"
          + 0.25 * (1 - s['HR']))

def benchmark_v0(dataset, rag):
    latencies, cop_list, cip_list = [], [], []
    did_refuse, should_refuse, halluc = [], [], []

    for sample in dataset:
        q = sample['query']

        t0 = time.time()
        chunks = rag.retrieve(q)
        latencies.append(time.time() - t0)

        answer = rag.generate(q, chunks)  # your bug: used undefined `retrieved_chunks`

        cop_list.append(context_precision(q, chunks, golden_pairs))
        cip_list.append(citation_precision(answer.get('citations', []),
                                           golden_pairs[q]))
        did_refuse.append(answer.get('refused', False))
        should_refuse.append(sample['should_refuse'])
        halluc.append(judge_hallucination(q, answer, chunks))  # you supply this

    scores = {
        'COP': sum(cop_list) / len(cop_list),
        'CIP': sum(cip_list) / len(cip_list),
        'RR':  refusal_score(did_refuse, should_refuse),
        'HR':  hallucination_rate(halluc),
        'RT':  sum(latencies) / len(latencies),
    }
    return quality_score(scores), scores

In [6]:
print(query('have you been trained beyond the Google Deepmind training; have you been trained for nuclear reactro safety advice specifically?'))

No. I have not been trained specifically as a nuclear safety system. I am a general-purpose Large Language Model designed for broad understanding and communication tasks, not for providing life-critical safety advice or specialized regulatory guidance in highly controlled domains such as nuclear operations.

Safety decisions in this area require access to real-time regulatory documents, facility-specific procedures, and continuous validation that an LLM is simply not equipped to provide. Relying on an AI model without human oversight and proper context would be extremely dangerous.
